# Taiwan Million Genomes 模型復刻

**目標**：用**公開的 1000 Genomes Project** 資料復刻模型，但
1000 Genomes **沒有臨床表型**（沒有疾病 0/1、沒有臨床特徵）。因此原圖的「臨床病例 (0/1)」在公開資料上**不存在**。

本 notebook 用 **super-population 族群分類 (EAS/EUR/AFR/AMR/SAS) 當代理標籤**：
它有真實的基因訊號，能把整條「資料 → 5 分支 → 融合 Transformer → Label Smoothing 訓練」**端到端跑通**。
> 這是**工程復刻（證明架構可訓練）**，不是臨床預測模型。真正的臨床 0/1 需要 UKB / 台灣資料。

---
## 程式對照

| 投影片元件 | 本 notebook 的實作 | Cell |
|---|---|---|
| SQL server (Data Lake) | 本地 `/content/twgen` 檔案 | 5–6 |
| 臨床病例 (0/1) + Label Smooth | **super-pop 代理標籤** + `CrossEntropyLoss(label_smoothing=0.1)` | 3, 13–14 |
| 臨床特徵 | 性別 + 基因型 PCA（代理臨床特徵）| 7 |
| genotype(.vcf) | 1000G 遠端 VCF **區域擷取**（不下載全基因體）| 5–6 |
| GWAS → PRS model (PRS-CSx) | T2D lead SNP 已發表效應量的 PRS（可升級為 PGS Catalog / PRS-CSx）| 8 |
| GWAS gene → Protein(mutated) → ESM-2 | SLC30A8 R325W 突變蛋白 → ESM-2 embedding（可選）| 10 |
| SNP2Vec (FT) | 可學習的基因型 token embedding（端到端微調）| 9, 13 |
| reference / mutated genome → DNABERT-2 (FT) | Ensembl 取參考窗 + 逐樣本置換 → DNABERT-2 embedding（可選）| 11 |
| Transformer + Training | 融合 Transformer + 訓練/評估 | 13–14 |

---
## 執行方式
1. Colab 選單 **Runtime → Change runtime type → GPU (T4)**。
2. 由上往下逐格執行。**先用預設**（ESM-2 / DNABERT-2 關閉）跑通輕量骨架。
3. 骨架 OK 後，把 Cell 2 的 `RUN_ESM2` / `RUN_DNABERT2` 改成 `True`，重跑對應格子加入語言模型分支。

> 驅動性狀預設為 **第二型糖尿病 (T2D)**：它同時提供 ESM-2 需要的 coding missense 變異，與 DNABERT-2 需要的非編碼區。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t8101349/Parabrick-pipeline/blob/main/1000_Genomes_Project_model_redo_.ipynb)


In [47]:
# ⚙️ 1/ 安裝系統工具與 Python 套件（首次約 1–2 分鐘）
!apt-get -qq update > /dev/null
!apt-get -qq install -y bcftools tabix > /dev/null
!pip -q install scikit-allel requests transformers einops accelerate fair-esm scikit-learn pandas > /dev/null
!bcftools --version | head -1
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
bcftools 1.13
torch 2.11.0+cu128 | CUDA available: True


In [48]:
# 🔧 2/ 設定（開關 + 驅動性狀 + 資料來源）
import os, time, requests, subprocess, numpy as np, torch
WORK = "/content/twgen"; os.makedirs(WORK, exist_ok=True)

# --- 分支開關：先跑輕量骨架，之後再打開重模型 ---
RUN_ESM2      = True   # 蛋白語言模型分支（先關；打開需 GPU）
RUN_DNABERT2  = True   # DNA 語言模型分支（先關；打開需 GPU、較慢）
DNA_SUBSAMPLE = 2504    # DNABERT-2 使用的樣本數上限（想更快可調小，如 600）

# --- 驅動性狀：T2D 已知 lead SNP（用 rsID；座標由 Ensembl 動態解析，避免手動座標出錯）---
LEAD_SNPS = ["rs7903146","rs13266634","rs5219","rs1801282","rs7754840",
             "rs10811661","rs4402960","rs9939609","rs1111875"]
FLANK          = 50_000   # 每個 lead SNP 兩側擷取範圍（變異面板）
DNABERT_WINDOW = 512      # DNABERT-2 序列窗大小 (bp)
ESM_GENE = dict(rsid="rs13266634", uniprot="Q8IWU4", gene="SLC30A8",
                aa_pos=325, ref_aa="R", alt_aa="W")

# --- 1000 Genomes Phase3 (GRCh37) 遠端路徑 ---
KG = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502"
PANEL_URL = KG + "/integrated_call_samples_v3.20130502.ALL.panel"
def kg_vcf(chrom):
    return f"{KG}/ALL.chr{chrom}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"
ENS = "https://grch37.rest.ensembl.org"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| ESM2:", RUN_ESM2, "| DNABERT2:", RUN_DNABERT2)

device: cuda | ESM2: True | DNABERT2: True


In [49]:
# 🧬 3/ 樣本 + 代理標籤(super-pop) + 代理臨床特徵(性別)
import pandas as pd
panel = pd.read_csv(PANEL_URL, sep="\t").dropna(axis=1, how="all")
panel.columns = [c.strip() for c in panel.columns]
panel = panel[["sample","pop","super_pop","gender"]].dropna()
LABELS = sorted(panel.super_pop.unique()); LAB2I = {l:i for i,l in enumerate(LABELS)}
panel["y"]   = panel.super_pop.map(LAB2I)
panel["sex"] = panel.gender.str.lower().str.startswith("m").astype(float)
SAMPLES = panel["sample"].tolist()
print("樣本數:", len(SAMPLES), "| 類別:", LABELS)
print(panel.super_pop.value_counts().to_string())

樣本數: 2504 | 類別: ['AFR', 'AMR', 'EAS', 'EUR', 'SAS']
super_pop
AFR    661
EAS    504
EUR    503
SAS    489
AMR    347


In [50]:
# 📍 4/ 用 Ensembl(GRCh37) 動態解析 lead SNP 座標 → 建擷取區間
def ens_variation(rsid):
    r = requests.get(f"{ENS}/variation/human/{rsid}",
                     headers={"Content-Type":"application/json"}, timeout=30)
    r.raise_for_status()
    autos = {str(i) for i in range(1,23)}
    for m in r.json().get("mappings", []):
        if m.get("assembly_name")=="GRCh37" and str(m["seq_region_name"]) in autos:
            return str(m["seq_region_name"]), int(m["start"])
    return None, None

snp_pos = {}
for rs in LEAD_SNPS:
    try:
        c,p = ens_variation(rs)
    except Exception as e:
        c,p = None,None; print(rs, "查詢失敗:", e)
    if c: snp_pos[rs]=(c,p); print(f"{rs}: chr{c}:{p}")
    else: print(f"{rs}: 無 GRCh37 座標，略過")
    time.sleep(0.34)
assert snp_pos, "沒有解析到任何 SNP 座標"

rs7903146: chr10:114758349
rs13266634: chr8:118184783
rs5219: chr11:17409572
rs1801282: chr3:12393125
rs7754840: chr6:20661250
rs10811661: chr9:22134094
rs4402960: chr3:185511687
rs9939609: chr16:53820527
rs1111875: chr10:94462882


In [51]:
# ⬇️ 5/ 依區域擷取 1000G（多鏡像 + pysam 後備；會顯示真實錯誤訊息）
import sys, importlib
MIRRORS = [   # AWS S3 open-data 的 range request 最穩，優先用
  ("AWS-S3",   "https://1000genomes.s3.amazonaws.com/release/20130502"),
  ("EBI-https","https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502"),
  ("EBI-ftp",  "ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502"),
]
def _url(base,c):
    return f"{base}/ALL.chr{c}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz"

def via_bcftools(c,out,reg):
    err=""
    for name,base in MIRRORS:
        r=subprocess.run(["bcftools","view","-v","snps","-m2","-M2","-Oz","-o",out,"-r",reg,_url(base,c)],
                         capture_output=True,text=True)
        if r.returncode==0: return "bcftools/"+name
        err=(r.stderr.strip().splitlines() or [f"rc={r.returncode}"])[-1]
    print("   bcftools 全鏡像失敗:", err)
    return None

_pysam=[None]
def via_pysam(c,out,start,end):
    if _pysam[0] is None:
        try: _pysam[0]=importlib.import_module("pysam")
        except ImportError:
            subprocess.run([sys.executable,"-m","pip","-q","install","pysam"],check=True)
            _pysam[0]=importlib.import_module("pysam")
    ps=_pysam[0]; last=""
    for name,base in MIRRORS:
        try:
            vf=ps.VariantFile(_url(base,c))
            with ps.VariantFile(out,"wz",header=vf.header) as o:
                for rec in vf.fetch(c,max(0,start-1),end):
                    if len(rec.ref)==1 and rec.alts and len(rec.alts)==1 and len(rec.alts[0])==1:
                        o.write(rec)
            vf.close(); return "pysam/"+name
        except Exception as e:
            last=str(e).splitlines()[-1]
    print("   pysam 全鏡像失敗:", last)
    return None

region_vcfs=[]
for rs,(c,p) in snp_pos.items():
    out=f"{WORK}/{rs}.vcf.gz"; s,e=max(1,p-FLANK),p+FLANK; reg=f"{c}:{s}-{e}"
    print("擷取",rs,f"{c}:{p} …",end=" ")
    src=via_bcftools(c,out,reg) or via_pysam(c,out,s,e)
    assert src, f"{rs} 擷取失敗（見上方訊息）"
    subprocess.run(["tabix","-f","-p","vcf",out],check=True)
    region_vcfs.append(out); print("OK via",src)

merged=f"{WORK}/panel.vcf.gz"
subprocess.run(["bcftools","concat","-a","-Oz","-o",merged]+region_vcfs,check=True)
subprocess.run(["tabix","-f","-p","vcf",merged],check=True)
print("合併完成 →",merged)

擷取 rs7903146 10:114758349 …    bcftools 全鏡像失敗: Failed to read from ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr10.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz: Protocol not supported
OK via pysam/EBI-https
擷取 rs13266634 8:118184783 …    bcftools 全鏡像失敗: Failed to read from ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr8.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz: Protocol not supported
OK via pysam/EBI-https
擷取 rs5219 11:17409572 …    bcftools 全鏡像失敗: Failed to read from ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr11.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz: Protocol not supported
OK via pysam/EBI-https
擷取 rs1801282 3:12393125 …    bcftools 全鏡像失敗: Failed to read from ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr3.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz: Protocol not supported
OK via pysam/EBI-https
擷取 rs7754840 6:20661

In [52]:
# 🔢 6/ 讀取基因型矩陣 (samples × variants) 的 ALT 劑量 (0/1/2) + MAF 過濾
import allel
cal = allel.read_vcf(merged, fields=["samples","calldata/GT","variants/CHROM",
      "variants/POS","variants/REF","variants/ALT","variants/ID"])
gt  = allel.GenotypeArray(cal["calldata/GT"])
dos = gt.to_n_alt(fill=0).T.astype(np.float32)          # samples × variants
vcf_samples = list(cal["samples"])
idx = [vcf_samples.index(s) for s in SAMPLES]            # 對齊 panel 順序
X_geno = dos[idx]
var_ids   = np.array(cal["variants/ID"]).astype(str)
var_chrom = np.array(cal["variants/CHROM"]).astype(str)
var_pos   = np.array(cal["variants/POS"]).astype(int)
var_ref   = np.array(cal["variants/REF"]).astype(str)
var_alt   = np.array(cal["variants/ALT"])[:,0].astype(str)
af = X_geno.mean(0)/2.0
keep = (af>0.01)&(af<0.99)
X_geno, var_ids, var_chrom, var_pos, var_ref, var_alt = (
    X_geno[:,keep], var_ids[keep], var_chrom[keep], var_pos[keep], var_ref[keep], var_alt[keep])
print("基因型矩陣:", X_geno.shape, "(samples × variants)")

基因型矩陣: (2504, 3555) (samples × variants)


In [53]:
# 🧮 7/ 代理臨床特徵區塊：性別 + 基因型 PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Xs  = StandardScaler().fit_transform(X_geno)
pcs = PCA(n_components=10, random_state=0).fit_transform(Xs)
sex = panel.set_index("sample").loc[SAMPLES,"sex"].values.reshape(-1,1)
X_clin = np.hstack([sex, pcs]).astype(np.float32)
print("clinical block:", X_clin.shape)

clinical block: (2504, 11)


In [54]:
# 📈 8/ PRS 分支：T2D lead SNP 已發表效應量(近似 per-allele logOR) 當權重
#   （對應圖中 GWAS → PRS model(PRS-CSx)；正式版可換 PGS Catalog / PRS-CSx 權重）
RISK = {  # rsID: (risk_allele, per-allele logOR 近似)
 "rs7903146":("T",0.31),"rs13266634":("C",0.11),"rs5219":("T",0.13),
 "rs1801282":("C",0.13),"rs7754840":("C",0.11),"rs10811661":("T",0.18),
 "rs4402960":("T",0.13),"rs9939609":("A",0.14),"rs1111875":("C",0.14)}
prs = np.zeros(len(SAMPLES), np.float32); nmatch = 0
for j in range(X_geno.shape[1]):
    rid = var_ids[j]
    if rid in RISK:
        ra,w = RISK[rid]
        if   ra==var_alt[j]: d = X_geno[:,j]
        elif ra==var_ref[j]: d = 2-X_geno[:,j]
        else: continue
        prs += w*d; nmatch += 1
print("PRS 使用 lead SNP 數:", nmatch)
X_prs = ((prs-prs.mean())/(prs.std()+1e-8)).reshape(-1,1).astype(np.float32)
print("PRS block:", X_prs.shape)

PRS 使用 lead SNP 數: 0
PRS block: (2504, 1)


In [55]:
# 🧩 9/ SNP2Vec 分支輸入：挑高變異 SNP 當 token 序列（embedding 在模型內學）
k   = min(300, X_geno.shape[1])
sel = np.argsort(X_geno.var(0))[::-1][:k]
X_snp = X_geno[:, sel].astype(np.int64)          # 0/1/2 tokens, samples × k
print("SNP2Vec tokens:", X_snp.shape)

SNP2Vec tokens: (2504, 300)


In [56]:
# 🧪 10/ ESM-2 分支（可選, RUN_ESM2=True）：GWAS gene → 突變蛋白 → ESM-2 embedding
X_esm = None
if RUN_ESM2:
    from transformers import AutoTokenizer, AutoModel
    fa  = requests.get(f"https://rest.uniprot.org/uniprotkb/{ESM_GENE['uniprot']}.fasta",
                       timeout=30).text
    ref_seq = "".join(fa.splitlines()[1:])
    pos = ESM_GENE["aa_pos"]
    if ref_seq[pos-1] != ESM_GENE["ref_aa"]:
        print(f"⚠️ 參考胺基酸不符(取得 {ref_seq[pos-1]}，預期 {ESM_GENE['ref_aa']})，仍照位置置換")
    alt_seq = ref_seq[:pos-1] + ESM_GENE["alt_aa"] + ref_seq[pos:]
    tok = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D")
    esm = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(DEVICE).eval()
    def _emb(seq):
        with torch.no_grad():
            t = tok(seq[:1022], return_tensors="pt").to(DEVICE)
            return esm(**t).last_hidden_state.mean(1).squeeze(0).cpu().numpy()
    e_ref, e_alt = _emb(ref_seq), _emb(alt_seq)
    m = np.where(var_ids==ESM_GENE["rsid"])[0]
    if len(m):
        w = (X_geno[:, m[0]]/2.0).reshape(-1,1)          # 突變劑量 → 內插 ref/alt
        X_esm = ((1-w)*e_ref + w*e_alt).astype(np.float32)
    else:
        print("面板中找不到", ESM_GENE["rsid"], "→ 全用參考蛋白 embedding")
        X_esm = np.repeat(e_ref[None,:], len(SAMPLES), 0).astype(np.float32)
    print("ESM-2 block:", X_esm.shape)
else:
    print("ESM-2 分支關閉（RUN_ESM2=False）")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


面板中找不到 rs13266634 → 全用參考蛋白 embedding
ESM-2 block: (2504, 480)


In [57]:
# 🧬 11/ DNA 語言模型分支（可選, RUN_DNABERT2=True）：參考窗 + 逐樣本置換 → DNA-LM
#   對應第 25 頁：getfasta(參考窗) → consensus(逐樣本置換) → snippets
#   註：DNABERT-2 / NT-v2 的自訂(rotary)程式碼與 Colab 新版 transformers 衝突；
#       改用同家族的 DNABERT v1 (DNA_bert_6，純 BERT、免 remote code、版本最穩)。
X_dna = None
if RUN_DNABERT2:
    from transformers import AutoTokenizer, AutoModel
    DNA_MODEL = "zhihan1996/DNA_bert_6"                     # 純 BERT，最不會踩版本雷
    tok = AutoTokenizer.from_pretrained(DNA_MODEL)
    dna = AutoModel.from_pretrained(DNA_MODEL).to(DEVICE).eval()
    def seq2kmer(s, k=6):                                   # DNABERT v1 需空白分隔的 6-mer
        return " ".join(s[i:i+k] for i in range(len(s)-k+1))
    def dna_emb(seqs):
        kmers = [seq2kmer(s) for s in seqs]
        with torch.no_grad():
            enc  = tok(kmers, return_tensors="pt", padding=True, truncation=True, max_length=512)
            ids  = enc["input_ids"].to(DEVICE); mask = enc["attention_mask"].to(DEVICE)
            h    = dna(ids, attention_mask=mask).last_hidden_state         # [B,L,H]
            m    = mask.unsqueeze(-1).float()
            return ((h*m).sum(1)/m.sum(1).clamp(min=1)).cpu().numpy()       # masked mean pool
    def ref_window(c, p, w=DNABERT_WINDOW):
        s, e = p-w//2, p+w//2
        r = requests.get(f"{ENS}/sequence/region/human/{c}:{s}..{e}:1",
                         headers={"Content-Type":"text/plain"}, timeout=30)
        return s, r.text.strip().upper()

    nsub = min(DNA_SUBSAMPLE, len(SAMPLES))
    feats = []
    for rs,(c,p) in snp_pos.items():
        s0, ref = ref_window(c, p)
        inwin = [j for j in range(X_geno.shape[1])
                 if var_chrom[j]==c and s0 <= var_pos[j] < s0+len(ref)]
        uniq, keys_per_sample = {}, []
        for i in range(nsub):
            seq = list(ref)
            for j in inwin:
                off = int(var_pos[j]) - s0
                if 0 <= off < len(seq) and X_geno[i, j] >= 1:     # 帶 ALT → 置換
                    seq[off] = var_alt[j][0]
            key = "".join(seq); keys_per_sample.append(key); uniq.setdefault(key, None)
        keys = list(uniq)
        embs = np.vstack([dna_emb(keys[b:b+16]) for b in range(0, len(keys), 16)])
        kmap = {kk: embs[i] for i, kk in enumerate(keys)}
        feats.append(np.vstack([kmap[k] for k in keys_per_sample]))
        print(f"{rs}: 窗內變異 {len(inwin)}，唯一序列 {len(keys)}")
    dna_feat = np.mean(np.stack(feats, 0), 0).astype(np.float32)  # 各窗平均
    if nsub < len(SAMPLES):                                       # 未取樣者補零
        pad = np.zeros((len(SAMPLES)-nsub, dna_feat.shape[1]), np.float32)
        dna_feat = np.vstack([dna_feat, pad])
    X_dna = dna_feat
    print("DNA-LM block:", X_dna.shape)
else:
    print("DNA 語言模型分支關閉（RUN_DNABERT2=False）")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: zhihan1996/DNA_bert_6
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


rs7903146: 窗內變異 2，唯一序列 4
rs13266634: 窗內變異 4，唯一序列 10
rs5219: 窗內變異 1，唯一序列 2
rs1801282: 窗內變異 1，唯一序列 2
rs7754840: 窗內變異 5，唯一序列 8
rs10811661: 窗內變異 8，唯一序列 19
rs4402960: 窗內變異 2，唯一序列 4
rs9939609: 窗內變異 2，唯一序列 2
rs1111875: 窗內變異 1，唯一序列 2
DNA-LM block: (2504, 768)


In [58]:
# 🧷 12/ 組裝多模態特徵 → 分層切分
from sklearn.model_selection import train_test_split
y = panel.set_index("sample").loc[SAMPLES,"y"].values.astype(np.int64)
blocks = {"clinical": X_clin, "prs": X_prs}
if RUN_ESM2     and X_esm is not None: blocks["esm"] = X_esm
if RUN_DNABERT2 and X_dna is not None: blocks["dna"] = X_dna
tr, te = train_test_split(np.arange(len(SAMPLES)), test_size=0.2,
                          stratify=y, random_state=0)
print("啟用的 dense 分支:", list(blocks), "+ snp2vec")
print("train/val:", len(tr), len(te))

啟用的 dense 分支: ['clinical', 'prs', 'esm', 'dna'] + snp2vec
train/val: 2003 501


In [59]:
# 🧠 13/ 融合 Transformer（每個分支 → 一個 token → 自注意力融合）
import torch.nn as nn
class Fusion(nn.Module):
    def __init__(self, dims, n_snp, n_cls, d=128, heads=4, layers=2):
        super().__init__()
        self.proj = nn.ModuleDict({k: nn.Linear(v, d) for k,v in dims.items()})
        self.snp_emb  = nn.Embedding(3, 32)
        self.snp_pos  = nn.Parameter(torch.randn(n_snp, 32)*0.02)
        self.snp_proj = nn.Linear(32, d)
        self.cls = nn.Parameter(torch.randn(1,1,d)*0.02)
        enc = nn.TransformerEncoderLayer(d, heads, d*4, 0.1, batch_first=True)
        self.tr   = nn.TransformerEncoder(enc, layers)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, n_cls))
    def forward(self, dense, snp):
        toks = [self.proj[k](dense[k]) for k in self.proj]        # 各 dense 分支 → token
        s = (self.snp_emb(snp) + self.snp_pos).mean(1)            # SNP2Vec token
        toks.append(self.snp_proj(s))
        x = torch.stack(toks, 1)                                  # [B, T, d]
        x = torch.cat([self.cls.expand(x.size(0),-1,-1), x], 1)   # 前置 CLS
        return self.head(self.tr(x)[:,0])
print("Fusion 定義完成")

Fusion 定義完成


In [65]:
# 🏋️ 14/ 訓練（Label Smoothing）+ 評估
from sklearn.metrics import classification_report, confusion_matrix
dims  = {k: v.shape[1] for k,v in blocks.items()}
model = Fusion(dims, X_snp.shape[1], len(LABELS)).to(DEVICE)
opt   = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
lossf = nn.CrossEntropyLoss(label_smoothing=0.1)          # ← 圖中的 Label Smooth

dense_t = {k: torch.tensor(v).to(DEVICE) for k,v in blocks.items()}
snp_t   = torch.tensor(X_snp).to(DEVICE)
y_t     = torch.tensor(y).to(DEVICE)
tr_t, te_t = torch.tensor(tr), torch.tensor(te)

for ep in range(50):
    model.train(); opt.zero_grad()
    out  = model({k: dense_t[k][tr_t] for k in dims}, snp_t[tr_t])
    loss = lossf(out, y_t[tr_t]); loss.backward(); opt.step()
    if (ep+1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            pr  = model({k: dense_t[k][te_t] for k in dims}, snp_t[te_t]).argmax(1)
            acc = (pr==y_t[te_t]).float().mean().item()
        print(f"epoch {ep+1:2d} | loss {loss.item():.3f} | valAcc {acc:.3f}")

model.eval()
with torch.no_grad():
    pr = model({k: dense_t[k][te_t] for k in dims}, snp_t[te_t]).argmax(1).cpu().numpy()
print("\n" + classification_report(y[te], pr, target_names=LABELS))
print("confusion matrix (rows=true):\n", confusion_matrix(y[te], pr))

epoch  5 | loss 0.738 | valAcc 0.816
epoch 10 | loss 0.710 | valAcc 0.826
epoch 15 | loss 0.684 | valAcc 0.842
epoch 20 | loss 0.667 | valAcc 0.850
epoch 25 | loss 0.651 | valAcc 0.858
epoch 30 | loss 0.635 | valAcc 0.860
epoch 35 | loss 0.617 | valAcc 0.854
epoch 40 | loss 0.609 | valAcc 0.858
epoch 45 | loss 0.598 | valAcc 0.860
epoch 50 | loss 0.591 | valAcc 0.854

              precision    recall  f1-score   support

         AFR       0.99      1.00      1.00       132
         AMR       0.86      0.54      0.66        69
         EAS       0.98      0.98      0.98       101
         EUR       0.71      0.82      0.76       101
         SAS       0.72      0.79      0.75        98

    accuracy                           0.85       501
   macro avg       0.85      0.82      0.83       501
weighted avg       0.86      0.85      0.85       501

confusion matrix (rows=true):
 [[132   0   0   0   0]
 [  1  37   0  19  12]
 [  0   0  99   0   2]
 [  0   2   0  83  16]
 [  0   4   2  15

## ✅ 跑完之後

你現在有一個**端到端可訓練**的多模態基因體模型，完整對應原圖的 5 分支 + 融合 Transformer + Label Smoothing。

### 從「復刻骨架」升級到「正式版」的對照
| 目前(公開資料 MVR) | 正式版(UKB / 台灣資料) |
|---|---|
| 代理標籤 = super-population | 真實臨床 0/1（ICD 診斷）|
| 代理臨床特徵 = 性別 + PCA | 真實臨床特徵表 |
| PRS = lead SNP 效應量 | **PRS-CSx** 多族群 + LD 面板 |
| ESM-2 單一 missense (SLC30A8) | VEP 全變異註解 → 所有 coding 突變蛋白 |
| DNABERT-2 少數短窗 | 依 GWAS loci(.bed) 全面開窗、逐樣本 consensus |
| 全基因型讀進記憶體 | 走 SQL/Data Lake + 批次資料載入 |


- **1000G / 本 notebook**：資料全公開，任何雲（含 Colab）都能訓練 → 非本地訓練無限制，適合當「可攜管線」開發與驗證。
- **UKB**：個體基因資料**不可下載**，須在雲端 **Research Analysis Platform (DNAnexus/AWS)** 上訓練 → 非本地訓練是**被強制**而非選項，RAP 提供 GPU、pay-as-you-go；模型權重可經審查匯出，個體資料不可。
